# ERA5 hourly data --> SCEPTER climate files
Note, ERA5 hourly data predicts very low runoff rates when we scale up to the annual / monthly mean. They're generally consistent with the P-E water balance, or P-PET, but many reasonably watered sites had <2 mm/month of runoff. 

To get around this issue, I added a flag to calculate runoff using the Budyko relationship based on time-mean P/PET and an omega=1.8. This gives us more reasonable runoff / infiltration values. 

Also note that for climate variables below, naming multiple variables in a list will require the mean of the variables to be used in the SCEPTER input: 
```
soilwater_layers = ["volumetric_soil_water_layer_1", "volumetric_soil_water_layer_2"]
soiltemperature_layers = ["soil_temperature_level_1", "soil_temperature_level_2"]
runoff_vars = ["runoff"]
```

In [1]:
# -------------------------------------------
# 
# generate and save climate input files 
# 
# -------------------------------------------

# Need the following 1d arrays --------------
# 
# [ time ]: decimal year 
# [ temperature ]: degrees C
# [ soil moisture ]: mm / m
# 
import os 

import geopandas as gpd
import fsspec
import icechunk 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.style as mplstyle
import numpy as np
import pandas as pd
import s3fs
import xarray as xr

from ew_workflows import clim_process_helperfxns as cph

# ----------------------------------------------------------------
# [ RUN NAMES ]
# exe_central1p5 ; exe_central2p0 ; exe_limBio1p5 ; exe_lowCCS1p5 ; 
# exe_lowCDR1p5 ; exe_noBiochar1p5 ; exe_noWeathering1p5 ; 
# exe_ref1p5 ; exe_techBehaviorChange1p5 ; exe_techBehaviorChange2p0 ;
# 
# [ RUNS TO COMPARE ]
# exe_central1p5 ; exe_noWeathering1p5 ; exe_techBehaviorChange1p5
#
# ----------------------------------------------------------------

# --- add crop area options (it will save multiple identical clim dirs,
#     but rock-application-specific files will be added later)
repeat_clim_for_crop_area = True
crop_area_frac_arr = np.round(np.linspace(0.1, 1, 10), 2) # from 10% to 100% in 10% increments
# -------------------------------------------------------------------

# *****************************************************************
# [ gcam site group for save name ]
RUNNAME = "exe_techBehaviorChange1p5"
deg_resolution = 1 
save_subdir = f'one-cell-per-region_{deg_resolution}deg/{RUNNAME}'
# *****************************************************************
crop_area_frac = '1.0'  # crop frac -- doesn't matter because we only pull the coords, which are same among all crop_area_frac options


# --- read in site data
datpath = f's3://carbonplan-carbon-removal/ew-workflows-data/gcam/apprate_calculated/one-cell-per-region/{RUNNAME}/{crop_area_frac}-cropFraction'
fn = f'annual_rockflx_densest-ag-region_{deg_resolution}deg.pkl'
dfx = pd.read_pickle(os.path.join(datpath, fn))
# get unique coords
df = dfx.drop_duplicates(subset=["lat", "lon"])

In [2]:
# --- set up save
nyears_repeat = 80   # repeat timeseries for all resolutions
savepath = "s3://carbonplan-carbon-removal/ew-workflows-data/scepter/clim"
subdir_rule = "combined"

# input file for var details -------------------
inputvar_details_fn = "/Users/tylerkukla/Documents/GitHub/iam-rtm-compare/era5/inputvar_details.txt"

# whether to save climate figure ---------------
save_clim_fig = True
subdir_rule = "combined"

In [3]:
# --- create site names
df = df.sort_values(["region", "lat", "lon"]).copy()
# create tuple column
df["latlon"] = list(zip(df["lat"], df["lon"]))
# assign per-region integer ID
df["X"] = (
    df.groupby("region")["latlon"]
      .transform(lambda s: pd.factorize(s)[0] + 1)
)
# final unique identifier
df["region_uid"] = df["region_nospaces"] + "_" + df["X"].astype(str)
# drop helper columns
df = df.drop(columns=["latlon", "X"])

# --- create site dictionary
df_unique = df.drop_duplicates("region_uid").copy()
# convert lon to 0-360
df_unique['lon360'] = df_unique['lon'] % 360
site_dict = {
    row["region_uid"]: {
        "xlat": row["lat"],
        "xlon": row["lon"],
        "name": row["region_uid"],
    }
    for _, row in df_unique.iterrows()
}

## Read in era5 clim data

In [4]:
# --- read in parquet data (for temperature)
spin_parquet = 's3://carbonplan-carbon-removal/ew-workflows-data/valdation-data/sampled_locations_updated.parquet'
rtgdf = gpd.read_parquet(spin_parquet)
# --- read in zarr
datpath = "s3://carbonplan-carbon-removal/ew-workflows-data/gcam/gcam-era5-zarr/"
year_range = [2000, 2024]
ann_datasets = []
for year in range(year_range[0], year_range[1]+1):
    tds = xr.open_zarr(os.path.join(datpath, f"era5_sites_{year}.zarr"))
    ann_datasets.append(tds)
ds = xr.concat(ann_datasets, dim="year")

In [5]:
# --- compute long-term mean climate variables directly from ds

# [1] runoff: (P - E) [m yr-1] → [mm month-1]
qrun_ltm = (
    ds['tp_annual_sum'].mean('year') - ds['e_annual_sum'].mean('year')
) * 1000 / 12

# [2] soil moisture: depth-weighted mean across layers [m3 m-3] → [mm m-1]
soilwater_ltm = (
    ds['swvl1_annual_mean'].mean('year') * 7 +
    ds['swvl2_annual_mean'].mean('year') * 21 +
    ds['swvl3_annual_mean'].mean('year') * 72
) / 100 * 1000

# [3] temperature: from rtgdf, aligned to ds via site_id ↔ reg_id [K] → [°C]
temp_by_regid = rtgdf.drop_duplicates('reg_id').set_index('reg_id')['ERA5_stl1'] - 273.15
temp_ltm = xr.DataArray(
    temp_by_regid.loc[ds.site_id.values].values,
    coords={'site': ds.site.values},
    dims='site'
)

# --- assemble dataset with variable names cph expects
ds_ltm = xr.Dataset({
    'runoff_mm_month': qrun_ltm,
    'soilwater_mm_meter': soilwater_ltm,
    'temperature_c': temp_ltm,
})

# --- assign meaningful site names: {region_nospaces}_{deg_resolution}
regid_to_sitename = df_unique.set_index('reg_id')['region_nospaces']
site_names = [f"{regid_to_sitename[sid]}_{deg_resolution}" for sid in ds.site_id.values]
ds_ltm = ds_ltm.assign_coords(site=('site', site_names))

# --- build ds_dict in yearly_ltm format (12 evenly-spaced monthly steps)
ds_dict = {
    'yearly_ltm': ds_ltm.expand_dims(
        time_years=list(np.linspace(0, 1 - 0.08333, 12))
    )
}


In [6]:
# --- save SCEPTER climate files, repeating LTM for 80 years
if repeat_clim_for_crop_area:
    for crop_area_frac in crop_area_frac_arr:
        save_maindir = os.path.join(savepath, save_subdir, f"{crop_area_frac}-cropFraction")
        cph.save_all_case_climfiles_as_txt(
            ds_dict,
            nyears_data=nyears_repeat,
            save_maindir=save_maindir,
            inputvar_details_fn=inputvar_details_fn,
            save_clim_fig=save_clim_fig,
            subdir_rule=subdir_rule,
            ltm_repeat=True,
        )
else:
    save_maindir = os.path.join(savepath, save_subdir)
    cph.save_all_case_climfiles_as_txt(
        ds_dict,
        nyears_data=nyears_repeat,
        save_maindir=save_maindir,
        inputvar_details_fn=inputvar_details_fn,
        save_clim_fig=save_clim_fig,
        subdir_rule=subdir_rule,
        ltm_repeat=True,
    )

Now saving: Africa_Eastern_1 -- yearly_ltm
Now saving: Africa_Northern_1 -- yearly_ltm
Now saving: Africa_Southern_1 -- yearly_ltm
Now saving: Africa_Western_1 -- yearly_ltm
Now saving: Argentina_1 -- yearly_ltm
Now saving: Australia_NZ_1 -- yearly_ltm
Now saving: Brazil_1 -- yearly_ltm
Now saving: Canada_1 -- yearly_ltm
Now saving: Central_America_and_Caribbean_1 -- yearly_ltm
Now saving: Central_Asia_1 -- yearly_ltm
Now saving: China_1 -- yearly_ltm
Now saving: Colombia_1 -- yearly_ltm
Now saving: EU-12_1 -- yearly_ltm
Now saving: EU-15_1 -- yearly_ltm
Now saving: Europe_Eastern_1 -- yearly_ltm
Now saving: Europe_Non_EU_1 -- yearly_ltm
Now saving: European_Free_Trade_Association_1 -- yearly_ltm
Now saving: India_1 -- yearly_ltm
Now saving: Indonesia_1 -- yearly_ltm
Now saving: Japan_1 -- yearly_ltm
Now saving: Mexico_1 -- yearly_ltm
Now saving: Middle_East_1 -- yearly_ltm
Now saving: Pakistan_1 -- yearly_ltm
Now saving: Russia_1 -- yearly_ltm
Now saving: South_Africa_1 -- yearly_ltm


## Scratch -----

In [ ]:
# ----